# Cybersecurity Risk Advisor Expert System

In [7]:
pip install experta

In [8]:
import collections
import collections.abc

# Compatibility patch for Experta on newer Python versions.
for name in ["Mapping", "MutableMapping", "Sequence", "Callable", "Hashable"]:
    if not hasattr(collections, name):
        setattr(collections, name, getattr(collections.abc, name))

from experta import Fact, KnowledgeEngine, Rule, MATCH, P

In [9]:
class SecurityIssue(Fact):
    pass

class LoginAttempts(Fact):
    pass

SEVERITY_WEIGHTS = {
    "Low": 25,
    "Medium": 50,
    "High": 75,
    "Critical": 100,
}

ISSUE_LABELS = {
    "weak_password": "Weak Password",
    "no_mfa": "No Multi-Factor Authentication",
    "outdated_os": "Outdated Operating System",
    "antivirus_disabled": "Antivirus Disabled",
    "suspicious_email_clicked": "Suspicious Email Clicked",
    "public_wifi_used": "Public Wi-Fi Used",
    "no_vpn": "No VPN",
    "ransomware_detected": "Ransomware Detected",
    "no_backup": "No Backup",
    "data_breach": "Data Breach",
    "open_rdp": "Open RDP Port",
    "low_security_awareness": "Low Security Awareness",
}

In [10]:
KNOWLEDGE_BASE = {
    "account_compromise": {
        "rule_name": "R1 - Account Compromise Risk",
        "risk": "Account Compromise Risk",
        "conditions": ["weak_password", "no_mfa"],
        "severity": "High",
        "rule_cf": 0.90,
        "advice": "Use strong unique passwords, enable MFA, and review account activity regularly.",
        "reason": "A weak password combined with missing MFA makes unauthorized access much easier.",
    },
    "malware_infection": {
        "rule_name": "R2 - Malware Infection Risk",
        "risk": "Malware Infection Risk",
        "conditions": ["outdated_os", "antivirus_disabled"],
        "severity": "High",
        "rule_cf": 0.85,
        "advice": "Update the operating system, enable antivirus protection, and run a full malware scan.",
        "reason": "An outdated system without active antivirus protection is more exposed to malware.",
    },
    "phishing_attack": {
        "rule_name": "R3 - Phishing Attack Risk",
        "risk": "Phishing Attack Risk",
        "conditions": ["suspicious_email_clicked", "low_security_awareness"],
        "severity": "Medium",
        "rule_cf": 0.80,
        "advice": "Reset affected passwords, report the email, and provide phishing awareness training.",
        "reason": "Clicking a suspicious email link with low awareness increases phishing success probability.",
    },
    "public_wifi": {
        "rule_name": "R4 - Public Wi-Fi Risk",
        "risk": "Public Wi-Fi Risk",
        "conditions": ["public_wifi_used", "no_vpn"],
        "severity": "Medium",
        "rule_cf": 0.75,
        "advice": "Avoid sensitive transactions on public Wi-Fi and use a trusted VPN.",
        "reason": "Using public Wi-Fi without VPN can expose traffic to interception.",
    },
    "brute_force": {
        "rule_name": "R5 - Brute Force Attack Risk",
        "risk": "Brute Force Attack Risk",
        "conditions": ["failed_login_attempts >= 5"],
        "severity": "High",
        "rule_cf": 0.85,
        "advice": "Enable account lockout, review login logs, use MFA, and block suspicious IP addresses.",
        "reason": "A high number of failed login attempts may indicate password guessing or automated attacks.",
    },
    "ransomware_data_loss": {
        "rule_name": "R6 - Ransomware and Data Loss Risk",
        "risk": "Ransomware and Data Loss Risk",
        "conditions": ["ransomware_detected", "no_backup"],
        "severity": "Critical",
        "rule_cf": 0.95,
        "advice": "Isolate affected systems, preserve evidence, restore from clean backups, and start incident response.",
        "reason": "Ransomware detection without reliable backups creates a serious data loss situation.",
    },
    "data_breach": {
        "rule_name": "R7 - Data Breach Risk",
        "risk": "Data Breach Risk",
        "conditions": ["data_breach"],
        "severity": "Critical",
        "rule_cf": 0.95,
        "advice": "Contain the incident, rotate credentials, notify responsible teams, and investigate exposed data.",
        "reason": "A reported data breach is a critical cybersecurity incident requiring immediate response.",
    },
    "remote_access_exposure": {
        "rule_name": "R8 - Remote Access Exposure Risk",
        "risk": "Remote Access Exposure Risk",
        "conditions": ["open_rdp", "weak_password"],
        "severity": "High",
        "rule_cf": 0.88,
        "advice": "Close public RDP access, restrict access by IP, enable VPN, and enforce strong authentication.",
        "reason": "Open RDP with weak passwords is a common path for unauthorized remote access.",
    },
}

In [11]:
def calculate_final_cf(rule_cf, *fact_cfs):
    if not fact_cfs:
        return round(rule_cf, 2)
    return round(rule_cf * min(fact_cfs), 2)


def classify_overall_risk(score):
    if score >= 75:
        return "Critical Risk"
    if score >= 50:
        return "High Risk"
    if score >= 25:
        return "Medium Risk"
    return "Low Risk"


def calculate_overall_score(risks):
    if not risks:
        return 10.0, "Low Risk"

    components = []
    for risk in risks:
        severity_weight = SEVERITY_WEIGHTS[risk["severity"]]
        components.append(severity_weight * risk["confidence"])

    score = max(components) * 0.70 + sum(components) * 0.30
    score = min(100.0, round(score, 2))
    return score, classify_overall_risk(score)

In [12]:
class CyberSecurityRiskAdvisor(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        self.detected_risks = []
        self.trace = []

    def _add_risk(self, key, *fact_cfs):
        item = KNOWLEDGE_BASE[key]
        confidence = calculate_final_cf(item["rule_cf"], *fact_cfs)

        risk_record = {
            "rule": item["rule_name"],
            "risk": item["risk"],
            "severity": item["severity"],
            "confidence": confidence,
            "reason": item["reason"],
            "advice": item["advice"],
            "conditions": item["conditions"],
        }

        self.detected_risks.append(risk_record)
        self.trace.append(
            f"{item['rule_name']} fired -> {item['risk']} | "
            f"Severity: {item['severity']} | Confidence: {confidence}"
        )

    @Rule(
        SecurityIssue(name="ransomware_detected", status=True, cf=MATCH.cf1),
        SecurityIssue(name="no_backup", status=True, cf=MATCH.cf2),
        salience=100,
    )
    def ransomware_and_data_loss_risk(self, cf1, cf2):
        self._add_risk("ransomware_data_loss", cf1, cf2)

    @Rule(
        SecurityIssue(name="data_breach", status=True, cf=MATCH.cf1),
        salience=95,
    )
    def data_breach_risk(self, cf1):
        self._add_risk("data_breach", cf1)

    @Rule(
        SecurityIssue(name="weak_password", status=True, cf=MATCH.cf1),
        SecurityIssue(name="no_mfa", status=True, cf=MATCH.cf2),
        salience=85,
    )
    def account_compromise_risk(self, cf1, cf2):
        self._add_risk("account_compromise", cf1, cf2)

    @Rule(
        SecurityIssue(name="outdated_os", status=True, cf=MATCH.cf1),
        SecurityIssue(name="antivirus_disabled", status=True, cf=MATCH.cf2),
        salience=80,
    )
    def malware_infection_risk(self, cf1, cf2):
        self._add_risk("malware_infection", cf1, cf2)

    @Rule(
        LoginAttempts(count=P(lambda value: value >= 5), cf=MATCH.cf1),
        salience=75,
    )
    def brute_force_attack_risk(self, cf1):
        self._add_risk("brute_force", cf1)

    @Rule(
        SecurityIssue(name="open_rdp", status=True, cf=MATCH.cf1),
        SecurityIssue(name="weak_password", status=True, cf=MATCH.cf2),
        salience=70,
    )
    def remote_access_exposure_risk(self, cf1, cf2):
        self._add_risk("remote_access_exposure", cf1, cf2)

    @Rule(
        SecurityIssue(name="suspicious_email_clicked", status=True, cf=MATCH.cf1),
        SecurityIssue(name="low_security_awareness", status=True, cf=MATCH.cf2),
        salience=60,
    )
    def phishing_attack_risk(self, cf1, cf2):
        self._add_risk("phishing_attack", cf1, cf2)

    @Rule(
        SecurityIssue(name="public_wifi_used", status=True, cf=MATCH.cf1),
        SecurityIssue(name="no_vpn", status=True, cf=MATCH.cf2),
        salience=55,
    )
    def public_wifi_risk(self, cf1, cf2):
        self._add_risk("public_wifi", cf1, cf2)

    def get_report(self):
        score, level = calculate_overall_score(self.detected_risks)

        if not self.trace:
            self.trace.append("No high-confidence rule fired. Current indicators show a low cybersecurity risk.")

        return {
            "overall_score": score,
            "overall_level": level,
            "detected_risks": self.detected_risks,
            "explanation_trace": self.trace,
        }


In [13]:
def analyze_cybersecurity_risk(issue_confidences, failed_login_attempts=0, failed_login_cf=0.80):
    engine = CyberSecurityRiskAdvisor()
    engine.reset()

    for issue_name, cf in issue_confidences.items():
        engine.declare(SecurityIssue(name=issue_name, status=True, cf=float(cf)))

    engine.declare(LoginAttempts(count=int(failed_login_attempts), cf=float(failed_login_cf)))
    engine.run()
    return engine.get_report()

In [14]:
def print_report(report):
    print("=" * 70)
    print("Cybersecurity Risk Advisor Report")
    print("=" * 70)
    print(f"Overall Risk Level: {report['overall_level']}")
    print(f"Risk Score: {report['overall_score']} / 100")
    print("\nDetected Risks:")

    if not report["detected_risks"]:
        print("- No major risk detected.")
    else:
        for index, risk in enumerate(report["detected_risks"], start=1):
            print(f"\n{index}. {risk['risk']}")
            print(f"   Rule: {risk['rule']}")
            print(f"   Severity: {risk['severity']}")
            print(f"   Confidence: {risk['confidence']}")
            print(f"   Why: {risk['reason']}")
            print(f"   Advice: {risk['advice']}")

    print("\nExplanation Trace:")
    for step in report["explanation_trace"]:
        print(f"- {step}")

## Knowledge Base Summary

In [15]:
for key, item in KNOWLEDGE_BASE.items():
    print(f"{item['rule_name']}")
    print(f"Risk: {item['risk']}")
    print(f"Conditions: {', '.join(item['conditions'])}")
    print(f"Severity: {item['severity']}")
    print(f"Rule CF: {item['rule_cf']}")
    print(f"Advice: {item['advice']}")
    print("-" * 70)

R1 - Account Compromise Risk
Risk: Account Compromise Risk
Conditions: weak_password, no_mfa
Severity: High
Rule CF: 0.9
Advice: Use strong unique passwords, enable MFA, and review account activity regularly.
----------------------------------------------------------------------
R2 - Malware Infection Risk
Risk: Malware Infection Risk
Conditions: outdated_os, antivirus_disabled
Severity: High
Rule CF: 0.85
Advice: Update the operating system, enable antivirus protection, and run a full malware scan.
----------------------------------------------------------------------
R3 - Phishing Attack Risk
Risk: Phishing Attack Risk
Conditions: suspicious_email_clicked, low_security_awareness
Severity: Medium
Rule CF: 0.8
Advice: Reset affected passwords, report the email, and provide phishing awareness training.
----------------------------------------------------------------------
R4 - Public Wi-Fi Risk
Risk: Public Wi-Fi Risk
Conditions: public_wifi_used, no_vpn
Severity: Medium
Rule CF: 0.75
A

## Example Test Case

In [16]:
engine = CyberSecurityRiskAdvisor()
engine.reset()
engine.declare(SecurityIssue(name="weak_password", status=True, cf=0.90))
engine.declare(SecurityIssue(name="no_mfa", status=True, cf=0.85))
engine.declare(LoginAttempts(count=7, cf=0.80))
engine.run()
manual_report = engine.get_report()
print_report(manual_report)

Cybersecurity Risk Advisor Report
Overall Risk Level: High Risk
Risk Score: 73.05 / 100

Detected Risks:

1. Account Compromise Risk
   Rule: R1 - Account Compromise Risk
   Severity: High
   Confidence: 0.77
   Why: A weak password combined with missing MFA makes unauthorized access much easier.
   Advice: Use strong unique passwords, enable MFA, and review account activity regularly.

2. Brute Force Attack Risk
   Rule: R5 - Brute Force Attack Risk
   Severity: High
   Confidence: 0.68
   Why: A high number of failed login attempts may indicate password guessing or automated attacks.
   Advice: Enable account lockout, review login logs, use MFA, and block suspicious IP addresses.

Explanation Trace:
- R1 - Account Compromise Risk fired -> Account Compromise Risk | Severity: High | Confidence: 0.77
- R5 - Brute Force Attack Risk fired -> Brute Force Attack Risk | Severity: High | Confidence: 0.68


In [17]:
#Low risk user
low_risk_report = analyze_cybersecurity_risk(
    issue_confidences={},
    failed_login_attempts=1,
    failed_login_cf=0.70
)
print_report(low_risk_report)

Cybersecurity Risk Advisor Report
Overall Risk Level: Low Risk
Risk Score: 10.0 / 100

Detected Risks:
- No major risk detected.

Explanation Trace:
- No high-confidence rule fired. Current indicators show a low cybersecurity risk.


In [18]:
# Medium risk user
medium_risk_report = analyze_cybersecurity_risk(
    issue_confidences={
        "suspicious_email_clicked": 0.80,
        "low_security_awareness": 0.75,
    },
    failed_login_attempts=2,
    failed_login_cf=0.70
)
print_report(medium_risk_report)

Cybersecurity Risk Advisor Report
Overall Risk Level: Medium Risk
Risk Score: 30.0 / 100

Detected Risks:

1. Phishing Attack Risk
   Rule: R3 - Phishing Attack Risk
   Severity: Medium
   Confidence: 0.6
   Why: Clicking a suspicious email link with low awareness increases phishing success probability.
   Advice: Reset affected passwords, report the email, and provide phishing awareness training.

Explanation Trace:
- R3 - Phishing Attack Risk fired -> Phishing Attack Risk | Severity: Medium | Confidence: 0.6


In [19]:
# High risk user
high_risk_report = analyze_cybersecurity_risk(
    issue_confidences={
        "weak_password": 0.90,
        "no_mfa": 0.85,
        "open_rdp": 0.80,
    },
    failed_login_attempts=8,
    failed_login_cf=0.85
)
print_report(high_risk_report)

Cybersecurity Risk Advisor Report
Overall Risk Level: Critical Risk
Risk Score: 89.7 / 100

Detected Risks:

1. Account Compromise Risk
   Rule: R1 - Account Compromise Risk
   Severity: High
   Confidence: 0.77
   Why: A weak password combined with missing MFA makes unauthorized access much easier.
   Advice: Use strong unique passwords, enable MFA, and review account activity regularly.

2. Brute Force Attack Risk
   Rule: R5 - Brute Force Attack Risk
   Severity: High
   Confidence: 0.72
   Why: A high number of failed login attempts may indicate password guessing or automated attacks.
   Advice: Enable account lockout, review login logs, use MFA, and block suspicious IP addresses.

3. Remote Access Exposure Risk
   Rule: R8 - Remote Access Exposure Risk
   Severity: High
   Confidence: 0.7
   Why: Open RDP with weak passwords is a common path for unauthorized remote access.
   Advice: Close public RDP access, restrict access by IP, enable VPN, and enforce strong authentication.

Ex

In [20]:
# Critical ransomware risk case
critical_risk_report = analyze_cybersecurity_risk(
    issue_confidences={
        "ransomware_detected": 0.95,
        "no_backup": 0.90,
        "data_breach": 0.85,
    },
    failed_login_attempts=10,
    failed_login_cf=0.80
)
print_report(critical_risk_report)

Cybersecurity Risk Advisor Report
Overall Risk Level: Critical Risk
Risk Score: 100.0 / 100

Detected Risks:

1. Ransomware and Data Loss Risk
   Rule: R6 - Ransomware and Data Loss Risk
   Severity: Critical
   Confidence: 0.85
   Why: Ransomware detection without reliable backups creates a serious data loss situation.
   Advice: Isolate affected systems, preserve evidence, restore from clean backups, and start incident response.

2. Data Breach Risk
   Rule: R7 - Data Breach Risk
   Severity: Critical
   Confidence: 0.81
   Why: A reported data breach is a critical cybersecurity incident requiring immediate response.
   Advice: Contain the incident, rotate credentials, notify responsible teams, and investigate exposed data.

3. Brute Force Attack Risk
   Rule: R5 - Brute Force Attack Risk
   Severity: High
   Confidence: 0.68
   Why: A high number of failed login attempts may indicate password guessing or automated attacks.
   Advice: Enable account lockout, review login logs, use MF

## Interactive Notebook Widget

In [22]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML
from html import escape

def safe_text(value):
    return escape(str(value))


def confidence_to_percent(value):
    try:
        if isinstance(value, str):
            value = value.replace("%", "").strip()
            value = float(value)
            if value <= 1:
                return value * 100
            return value

        value = float(value)
        if value <= 1:
            return value * 100
        return value
    except:
        return 0


def get_level_color(level):
    level = str(level).lower()

    if "critical" in level:
        return "#dc2626"
    if "high" in level:
        return "#ea580c"
    if "medium" in level:
        return "#ca8a04"
    if "low" in level:
        return "#16a34a"

    return "#334155"


def get_severity_color(severity):
    severity = str(severity).lower()

    if "critical" in severity:
        return "#dc2626"
    if "high" in severity:
        return "#ea580c"
    if "medium" in severity:
        return "#ca8a04"
    if "low" in severity:
        return "#16a34a"

    return "#334155"


def render_score_bar(score, level):
    color = get_level_color(level)

    return f"""
    <div style="margin-top:10px;">
        <div style="background:#e5e7eb;border-radius:12px;height:22px;width:100%;overflow:hidden;">
            <div style="background:{color};height:22px;width:{score}%;border-radius:12px;text-align:center;color:white;font-weight:bold;font-size:13px;">
                {score}/100
            </div>
        </div>
    </div>
    """


def render_confidence_bar(confidence):
    percent = confidence_to_percent(confidence)
    percent = max(0, min(100, percent))

    return f"""
    <div style="background:#e5e7eb;border-radius:10px;height:18px;width:220px;overflow:hidden;margin-top:4px;">
        <div style="background:#2563eb;height:18px;width:{percent}%;border-radius:10px;text-align:center;color:white;font-size:12px;">
            {percent:.1f}%
        </div>
    </div>
    """

display(HTML("""
<style>
.risk-panel {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-radius: 14px;
    padding: 18px;
    margin-bottom: 15px;
}

.section-title {
    font-size: 18px;
    font-weight: 700;
    color: #0f172a;
    margin-top: 12px;
    margin-bottom: 8px;
}

.result-card {
    background: white;
    border: 1px solid #e5e7eb;
    border-left: 6px solid #2563eb;
    border-radius: 14px;
    padding: 16px;
    margin: 12px 0;
    box-shadow: 0 2px 8px rgba(15, 23, 42, 0.05);
}

.small-note {
    color: #64748b;
    font-size: 13px;
}
</style>
"""))


ISSUE_CATEGORIES = {
    "Account & Identity Security": [
        "weak_password",
        "no_mfa",
        "low_security_awareness",
    ],
    "System & Malware Protection": [
        "outdated_os",
        "antivirus_disabled",
        "ransomware_detected",
        "no_backup",
    ],
    "Email & Phishing Security": [
        "suspicious_email_clicked",
    ],
    "Network & Remote Access Security": [
        "public_wifi_used",
        "no_vpn",
        "open_rdp",
    ],
    "Data Protection": [
        "data_breach",
    ],
}

issue_controls = {}
cf_controls = {}

for issue_name, label in ISSUE_LABELS.items():
    checkbox = widgets.Checkbox(
        value=False,
        description=label,
        indent=False,
        layout=widgets.Layout(width="310px")
    )

    slider = widgets.FloatSlider(
        value=0.80,
        min=0.0,
        max=1.0,
        step=0.05,
        description="Confidence",
        continuous_update=False,
        readout_format=".2f",
        disabled=True,
        style={"description_width": "85px"},
        layout=widgets.Layout(width="370px")
    )

    issue_controls[issue_name] = checkbox
    cf_controls[issue_name] = slider


def connect_checkbox_to_slider(issue_name):
    def toggle_slider(change):
        cf_controls[issue_name].disabled = not change["new"]

    issue_controls[issue_name].observe(toggle_slider, names="value")


for issue_name in ISSUE_LABELS:
    connect_checkbox_to_slider(issue_name)


failed_login_input = widgets.BoundedIntText(
    value=0,
    min=0,
    max=100,
    description="Failed Logins:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="230px")
)

failed_login_cf_input = widgets.FloatSlider(
    value=0.85,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Login Confidence",
    continuous_update=False,
    readout_format=".2f",
    disabled=True,
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px")
)


def toggle_failed_login_cf(change):
    failed_login_cf_input.disabled = change["new"] <= 0


failed_login_input.observe(toggle_failed_login_cf, names="value")


scenario_dropdown = widgets.Dropdown(
    options=[
        ("Custom Scenario", "custom"),
        ("Low Risk User", "low"),
        ("Medium Risk User", "medium"),
        ("High Risk User", "high"),
        ("Critical Ransomware Case", "critical"),
    ],
    value="custom",
    description="Scenario:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="360px")
)


def reset_inputs():
    for issue_name in ISSUE_LABELS:
        issue_controls[issue_name].value = False
        cf_controls[issue_name].value = 0.80
        cf_controls[issue_name].disabled = True

    failed_login_input.value = 0
    failed_login_cf_input.value = 0.85
    failed_login_cf_input.disabled = True


def apply_scenario(scenario):
    reset_inputs()

    if scenario == "low":
        selected = {
            "low_security_awareness": 0.55,
        }
        failed_login_input.value = 1
        failed_login_cf_input.value = 0.60

    elif scenario == "medium":
        selected = {
            "weak_password": 0.75,
            "no_mfa": 0.80,
            "public_wifi_used": 0.70,
        }
        failed_login_input.value = 3
        failed_login_cf_input.value = 0.75

    elif scenario == "high":
        selected = {
            "weak_password": 0.90,
            "no_mfa": 0.85,
            "outdated_os": 0.80,
            "antivirus_disabled": 0.85,
            "suspicious_email_clicked": 0.80,
            "open_rdp": 0.75,
        }
        failed_login_input.value = 8
        failed_login_cf_input.value = 0.85

    elif scenario == "critical":
        selected = {
            "ransomware_detected": 0.95,
            "no_backup": 0.90,
            "data_breach": 0.90,
            "open_rdp": 0.85,
            "antivirus_disabled": 0.80,
        }
        failed_login_input.value = 15
        failed_login_cf_input.value = 0.90

    else:
        selected = {}

    for issue_name, cf_value in selected.items():
        if issue_name in issue_controls:
            issue_controls[issue_name].value = True
            cf_controls[issue_name].value = cf_value
            cf_controls[issue_name].disabled = False

    failed_login_cf_input.disabled = failed_login_input.value <= 0


def on_scenario_changed(change):
    apply_scenario(change["new"])


scenario_dropdown.observe(on_scenario_changed, names="value")


analyze_button = widgets.Button(
    description="Analyze Cybersecurity Risk",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="260px", height="40px")
)

reset_button = widgets.Button(
    description="Reset Inputs",
    button_style="warning",
    icon="refresh",
    layout=widgets.Layout(width="150px", height="40px")
)

output_area = widgets.Output()


def on_reset_clicked(button):
    scenario_dropdown.value = "custom"
    reset_inputs()

    with output_area:
        clear_output()
        display(Markdown("### Inputs have been reset."))


reset_button.on_click(on_reset_clicked)


category_boxes = []

for category_name, issues in ISSUE_CATEGORIES.items():
    rows = []

    for issue_name in issues:
        if issue_name in ISSUE_LABELS:
            rows.append(
                widgets.HBox([
                    issue_controls[issue_name],
                    cf_controls[issue_name]
                ])
            )

    if rows:
        category_boxes.append(
            widgets.VBox([
                widgets.HTML(f"<div class='section-title'>{category_name}</div>"),
                *rows
            ])
        )

categorized_issues = []
for issues in ISSUE_CATEGORIES.values():
    categorized_issues.extend(issues)

other_rows = []
for issue_name in ISSUE_LABELS:
    if issue_name not in categorized_issues:
        other_rows.append(
            widgets.HBox([
                issue_controls[issue_name],
                cf_controls[issue_name]
            ])
        )

if other_rows:
    category_boxes.append(
        widgets.VBox([
            widgets.HTML("<div class='section-title'>Other Security Indicators</div>"),
            *other_rows
        ])
    )


ui = widgets.VBox([
    widgets.HTML("""
    <div class='risk-panel'>
        <h2 style='margin-bottom:4px;'>Cybersecurity Risk Advisor Interactive Panel</h2>
        <p class='small-note'>
        Select the cybersecurity issues that exist in the case, then adjust the confidence value for each selected issue.
        The system will use Experta rules, salience, and Certainty Factor to classify the final risk.
        </p>
    </div>
    """),

    widgets.HBox([scenario_dropdown]),

    *category_boxes,

    widgets.HTML("<div class='section-title'>Login Attack Indicator</div>"),
    widgets.HBox([failed_login_input, failed_login_cf_input]),

    widgets.HBox([analyze_button, reset_button]),

    output_area
])


def on_analyze_clicked(button):
    selected_issues = {}

    for issue_name, checkbox in issue_controls.items():
        if checkbox.value:
            selected_issues[issue_name] = cf_controls[issue_name].value

    report = analyze_cybersecurity_risk(
        selected_issues,
        failed_login_attempts=failed_login_input.value,
        failed_login_cf=failed_login_cf_input.value,
    )

    with output_area:
        clear_output()

        level = report.get("overall_level", "Unknown")
        score = report.get("overall_score", 0)

        try:
            score = int(round(float(score)))
        except:
            score = 0

        score = max(0, min(100, score))
        level_color = get_level_color(level)

        display(HTML(f"""
        <div class="risk-panel">
            <h2 style="margin-bottom:6px;">Analysis Result</h2>

            <div style="display:flex;gap:15px;flex-wrap:wrap;margin-top:10px;">
                <div style="background:white;border-radius:14px;padding:14px 18px;border:1px solid #e5e7eb;min-width:220px;">
                    <div style="font-size:13px;color:#64748b;">Overall Risk Level</div>
                    <div style="font-size:26px;font-weight:800;color:{level_color};">{safe_text(level)}</div>
                </div>

                <div style="background:white;border-radius:14px;padding:14px 18px;border:1px solid #e5e7eb;min-width:220px;">
                    <div style="font-size:13px;color:#64748b;">Risk Score</div>
                    <div style="font-size:26px;font-weight:800;color:#0f172a;">{score}/100</div>
                </div>

                <div style="background:white;border-radius:14px;padding:14px 18px;border:1px solid #e5e7eb;min-width:220px;">
                    <div style="font-size:13px;color:#64748b;">Detected Risks</div>
                    <div style="font-size:26px;font-weight:800;color:#0f172a;">{len(report.get("detected_risks", []))}</div>
                </div>
            </div>

            {render_score_bar(score, level)}
        </div>
        """))

        detected_risks = report.get("detected_risks", [])

        if not detected_risks:
            display(HTML("""
            <div class="result-card" style="border-left-color:#16a34a;">
                <h3 style="color:#16a34a;margin-top:0;">No Major Risk Detected</h3>
                <p>The selected indicators did not trigger any major cybersecurity risk rule.</p>
            </div>
            """))
        else:
            display(Markdown("## Detected Risks and Recommendations"))

            for risk in detected_risks:
                risk_name = risk.get("risk", "Unknown Risk")
                rule_name = risk.get("rule", "Unknown Rule")
                severity = risk.get("severity", "Unknown")
                confidence = risk.get("confidence", "N/A")
                reason = risk.get("reason", "No reason provided.")
                advice = risk.get("advice", "No advice provided.")

                severity_color = get_severity_color(severity)
                confidence_bar = render_confidence_bar(confidence)

                display(HTML(f"""
                <div class="result-card" style="border-left-color:{severity_color};">
                    <h3 style="margin-top:0;color:#0f172a;">{safe_text(risk_name)}</h3>

                    <p>
                        <b>Rule Fired:</b> {safe_text(rule_name)}<br>
                        <b>Severity:</b>
                        <span style="color:{severity_color};font-weight:700;">{safe_text(severity)}</span><br>
                        <b>Confidence:</b> {safe_text(confidence)}
                    </p>

                    {confidence_bar}

                    <p><b>Why this risk was detected:</b><br>{safe_text(reason)}</p>

                    <p><b>Recommended Action:</b><br>{safe_text(advice)}</p>
                </div>
                """))

        trace = report.get("explanation_trace", [])

        display(Markdown("## Explanation Trace"))

        if trace:
            trace_text = "\n".join([f"- {step}" for step in trace])
            display(Markdown(trace_text))
        else:
            display(Markdown("- No rules fired in this scenario."))


analyze_button.on_click(on_analyze_clicked)

display(ui)